In [10]:
from openai import OpenAI
import json
import os
with open("../../env/keys.json", "r", encoding="utf-8") as f:
    keys = json.load(f)

key = keys["OPENAI_SNCF"]
os.environ["OPENAI_API_KEY"] = key
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [29]:
with open("../../incidents_listes/incidents_categories.json", "r", encoding="utf-8") as f:
    incident_data = json.load(f)

with open("../../incidents_listes/incidents_arbo_simple.json", "r", encoding="utf-8") as f:
    incident_arbo = json.load(f)

with open("../../incidents_listes/incidents_arbo_comp.json", "r", encoding="utf-8") as f:
    incident_arbo = json.load(f)

In [57]:
message1 = "La lunette des toilettes glisse dans le wagon 3."

message2 = "Alors dans la salle, l'accessoire casier des bagages au niveau de l'escalier à gauche du wagon 6, il y a un tag"

message3 = "La tablette du siège 53 du wagon 2 est cassée."

message4 = "le siège 54 voitures 12 est cassée"


In [58]:
message = message3

In [18]:
arbo_str = json.dumps(incident_arbo, indent=2, ensure_ascii=False)

prompt = f"""
Tu es un assistant SNCF chargé d’analyser des incidents à partir de transcriptions audio.

Tu vas recevoir l’arborescence complète des incidents, organisée selon les niveaux suivants :
- Localisation
- Catégorie
- Objet
- Nature du problème

Ta tâche est de **mémoriser fidèlement** cette arborescence, en respectant **les termes exacts** fournis. Elle te servira ultérieurement à classer et structurer des incidents à partir de textes.

Voici l’arborescence :
{arbo_str}

⚠️ Ne réalise aucune analyse à ce stade.

**Réponds uniquement par la chaîne de caractères exacte suivante :**  
```plaintext
Structure comprise.
"""

tools = [
    {
        "type": "function",
        "function": {
            "name": "memorize_incident_tree",
            "description": "Retient l’arborescence des incidents pour analyse future.",
            "parameters": {
                "type": "object",
                "properties": {
                    "incident_tree": {
                        "type": "object",
                        "description": "Arborescence des incidents"
                    }
                },
                "required": ["incident_tree"]
            }
        }
    }
]

response = client.chat.completions.create(
    model="gpt-4.1",  # ou "gpt-4.1" si tu l’utilises
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    tools=tools,
    tool_choice="auto"
)

print(response.choices[0].message.content)


None


In [30]:
def get_all_keys_recursive(d):
    """Récupère tous les niveaux de l'arborescence sous forme de listes uniques."""
    keys_level_1 = list(d.keys())
    keys_level_2 = list({k for v in d.values() for k in v.keys()})
    keys_level_3 = list({k for v in d.values() for sub in v.values() for k in sub.keys()})
    #keys_level_4 = list({item for v in d.values() for sub in v.values() for val in sub.values() for item in val})
    return keys_level_1, keys_level_2, keys_level_3

# Récupérer toutes les options possibles
all_localisations, all_categories, all_objets = get_all_keys_recursive(incident_arbo)

def normalize_to_list(value):
    """Transforme une chaîne séparée par ; en liste, ou retourne la liste telle quelle."""
    if isinstance(value, str):
        return [v.strip() for v in value.split(";")]
    elif isinstance(value, list):
        return value
    else:
        return []


In [59]:
response = client.chat.completions.create(
    model="gpt-4-1106-preview",
    messages=[
        {"role": "system", "content": "Tu es un assistant chargé d'analyser des transcriptions audio d'agents SNCF pour identifier la localisation de l'incident."},
        {"role": "user", "content": f"""Voici les localisations possibles :
{incident_data["Localisation (QR)"]}

Ta tâche : extraire les localisations pertinentes dans la transcription suivante :
{message}

⚠️ Réponds uniquement par une liste exacte issue des localisations fournies, séparée par des points-virgules (;).
Si tu n'es pas sûr, écris : Je ne sais pas."""}
    ],
    temperature=0
)

localisation = response.choices[0].message.content


In [63]:
if "sais pas" in localisation:
    localisation = all_localisations

# Étape 2 : Catégorie
categories_possibles = set()
localisations = normalize_to_list(localisation)

for loc in localisations:
    loc = loc.strip()
    if loc in incident_arbo:
        categories_possibles.update(incident_arbo[loc].keys())
    else:
        #print(f"Localisation '{loc}' non trouvée dans l'arborescence.")
        categories_possibles = set(incident_arbo.get(loc, {}).keys())
# if isinstance(localisation, list):
#     for loc in localisation:
#         categories_possibles.update(incident_arbo.get(loc, {}).keys())
# else:
#     categories_possibles = set(incident_arbo.get(localisation, {}).keys())

print(categories_possibles)

{'Accessoires/ Environnement', 'Pack inoui', 'Info/Communication', 'Equipements SECURITE', 'Habillage', 'Eclairage', 'Prise 220 Volts', 'Siège'}


In [64]:
categories_possibles_str = "; ".join(sorted(categories_possibles))

response = client.chat.completions.create(
    model="gpt-4-1106-preview",
    messages=[
        {"role": "system", "content": "Tu es un assistant chargé d'analyser des transcriptions audio d'agents SNCF pour identifier la catégorie de l'incident."},
        {"role": "user", "content": f"""Voici les catégories possibles :
{categories_possibles_str}

Transcription :
{message}

⚠️ Réponds uniquement par une liste exacte issue des catégories fournies, séparée par des points-virgules (;).
Si tu n'es pas sûr, écris : Je ne sais pas."""}
    ],
    temperature=0
)

categorie = response.choices[0].message.content


In [65]:

if "sais pas" in categorie:
    categorie = all_categories

categories = normalize_to_list(categorie)

# Étape 3 : Objet
objets_possibles = set()
for loc in localisations:
    for cat in categories:
        if cat in incident_arbo.get(loc, {}):
            objets_possibles.update(incident_arbo[loc][cat].keys())
        #else:
            #print(f"Catégorie '{cat}' non trouvée sous localisation '{loc}' dans l'arborescence.")

# if isinstance(localisation, list) or isinstance(categorie, list):
#     for loc in localisation if isinstance(localisation, list) else [localisation]:
#         for cat in categorie if isinstance(categorie, list) else [categorie]:
#             objets_possibles.update(incident_arbo.get(loc, {}).get(cat, {}).keys())
# else:
#     objets_possibles = set(incident_arbo.get(localisation, {}).get(categorie, {}).keys())

print(objets_possibles)

{'Mécanisme pivotement (PSH)', 'Séparateur de place ', 'Accoudoir', 'Mécanisme inclinaison/réglage', 'Bouton siège pivotant (PSH)', 'Dossier', 'Assise', 'Repose tête', 'Bâti'}


In [66]:
objets_possibles_str = "; ".join(sorted(objets_possibles))

response = client.chat.completions.create(
    model="gpt-4-1106-preview",
    messages=[
        {"role": "system", "content": "Tu es un assistant chargé d'analyser des transcriptions audio d'agents SNCF pour identifier l'objet concerné par l'incident."},
        {"role": "user", "content": f"""Voici les objets possibles :
{objets_possibles_str}

Transcription :
{message}

⚠️ Réponds uniquement par une liste exacte issue des objets fournis, séparée par des points-virgules (;).
Si tu n'es pas sûr, écris : Je ne sais pas."""}
    ],
    temperature=0
)

objet = response.choices[0].message.content


In [67]:


if "sais pas" in objet:
    objet = all_objets
print(objet)
# Étape 4 : Nature du problème
print(incident_arbo)
problemes_possibles = set()

for loc in localisations:
    for cat in categories:
        for obj in objet:
            niveau_objet = incident_arbo.get(loc, {}).get(cat, {}).get(obj)

            if niveau_objet is None:
                continue

            # Si c'est une liste (souvent une liste de dicts)
            if isinstance(niveau_objet, list):
                for element in niveau_objet:
                    if isinstance(element, dict):
                        problemes_possibles.update(element.keys())
            # Si c'est un dict directement
            elif isinstance(niveau_objet, dict):
                problemes_possibles.update(niveau_objet.keys())




["Ecoulement de l'eau", 'Eclairage latéral (au dessus des vitres)', 'Distributeur de papier toilette', 'Livret, Guide de dépannage', 'Assise', 'Patte à cadenas', 'Rideau métallique', 'Table vis-à-vis', 'Frigo 38 litres', 'Meuble 4 portes', "Boucle d'arrimage", 'Portant à vêtements', "Bac d'écoulement", 'Balisage', 'Bandeau InOUI FS', 'Nez de marches', 'Vitrine réfrigérée', 'Meuble poubelle', 'Grand vantail', 'Baguette recouvrement', 'Eclairage blason (entre les vitres)', 'Trappe Sono/Interph', 'Trousse rouge médicalisée', 'Fleche 2', 'Repose pieds', 'LE BAR', 'Clayette supérieure', 'Prise USB', 'Vers de nvx horizons', 'Prenez de la hauteur', 'Réfrigération', 'Trappe savonnier', 'Es. Vélo-Plateforme', 'Ensemble cuvette', 'Miroir fond de salle', 'Savonnier', 'Pelliculage inférieur', 'Mon beau miroir', 'Patère', 'Liseuse', 'Porte gobelet', 'Barre montoire', 'Tabouret', 'Tablette mange debout', 'Détecteur de fumée', 'Poubelle', 'Séparateur de place ', 'Serrure anti-panique', 'ARS (Afficheu

In [68]:
problemes_possibles_str = "; ".join(sorted(problemes_possibles))

response = client.chat.completions.create(
    model="gpt-4-1106-preview",
    messages=[
        {"role": "system", "content": "Tu es un assistant chargé d'analyser des transcriptions audio d'agents SNCF pour identifier la nature exacte de l'incident."},
        {"role": "user", "content": f"""Contexte :
- Localisation : {localisation}
- Catégorie : {categorie}
- Objet : {objet}

Voici les natures de problème possibles :
{problemes_possibles_str}

Transcription :
{message}

⚠️ Réponds uniquement par une liste exacte issue des problèmes fournis, séparée par des points-virgules (;).
Si tu n'es pas sûr, écris : Je ne sais pas."""}
    ],
    temperature=0
)

probleme = response.choices[0].message.content


In [69]:
print("Localisation : ", localisation)
print("Catégorie : ", categorie)
print("Objet : ", objet)
print("Problème : ", probleme)


Localisation :  Place; Compartiment
Catégorie :  Siège
Objet :  ["Ecoulement de l'eau", 'Eclairage latéral (au dessus des vitres)', 'Distributeur de papier toilette', 'Livret, Guide de dépannage', 'Assise', 'Patte à cadenas', 'Rideau métallique', 'Table vis-à-vis', 'Frigo 38 litres', 'Meuble 4 portes', "Boucle d'arrimage", 'Portant à vêtements', "Bac d'écoulement", 'Balisage', 'Bandeau InOUI FS', 'Nez de marches', 'Vitrine réfrigérée', 'Meuble poubelle', 'Grand vantail', 'Baguette recouvrement', 'Eclairage blason (entre les vitres)', 'Trappe Sono/Interph', 'Trousse rouge médicalisée', 'Fleche 2', 'Repose pieds', 'LE BAR', 'Clayette supérieure', 'Prise USB', 'Vers de nvx horizons', 'Prenez de la hauteur', 'Réfrigération', 'Trappe savonnier', 'Es. Vélo-Plateforme', 'Ensemble cuvette', 'Miroir fond de salle', 'Savonnier', 'Pelliculage inférieur', 'Mon beau miroir', 'Patère', 'Liseuse', 'Porte gobelet', 'Barre montoire', 'Tabouret', 'Tablette mange debout', 'Détecteur de fumée', 'Poubelle'